# Medicinal Plant Classification: Local Machine Training & Evaluation

This notebook implements the complete end-to-end pipeline on your **local machine** using your **local GPU** (or CPU fallback).

### Pipeline Stages (Matching the Flowchart):
1. **Image Collection & Dataset Verification** (150 plant species in `dataset/`)
2. **Dataset Preparation** (Stratified 85/15 split, byte-level MD5 deduplication to prevent data leakage)
3. **Image Preprocessing & Leaf Segmentation** (Noise filtering, HSV green masking, morphological operations, GrabCut)
4. **Feature Extraction** (Shape, Size, Colour, Vein, Texture -> 36 quantitative features)
5. **MOGEO Optimization** (Multi-Objective Golden Eagle Optimizer for feature selection & CNN hyperparameters)
6. **CNN Layers (Model Training & Classification)** (Hybrid model fusing deep visual features with hand-crafted features)
7. **Performance Evaluation** (Accuracy, Precision, Recall, F1-Score, Confusion Matrix on untouched test set)
8. **Predict Plant Species & Associated Medicinal Properties** (Inference + botanical database lookup in `properties.csv`)

## 1. Local Environment & GPU Verification

In [ ]:
import os
import sys
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Device Capability: {torch.cuda.get_device_capability(0)}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("Running on Local CPU (CUDA is not currently enabled for PyTorch).")

## 2. Dataset Verification (150 Plant Species Classes)

In [ ]:
dataset_dir = "dataset"
classes = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))]
print(f"Total plant classes found: {len(classes)}")
assert len(classes) == 150, f"Expected 150 classes, found {len(classes)}"
print("Sample species:", classes[:5])

## 3. Dataset Preparation & Leakage-Free Splitting
Partitions the dataset into 85% development (split into `dev_train` and `dev_val`) and 15% `final_test`.
Performs MD5 hash checks to reassign duplicates to `dev_train` so test metrics reflect true generalization.

In [ ]:
from pipeline.data import build_manifest
from pipeline.utils import leakage_report

df = build_manifest('dataset', 'splits')
print(leakage_report(df.to_dict('records')))
print("\nSamples per split:")
print(df['split'].value_counts())

## 4. Preprocessing, Segmentation & Feature Extraction
Extracts 36 quantitative features across Shape, Size, Colour, Vein, and Texture, and saves segmented leaves to `preprocessed_dataset/`.

In [ ]:
# Run preprocessing and feature extraction
!python run_preprocess.py --dataset-dir dataset --output-dir . --skip-existing-csv

## 5. Baseline CNN Model Training
Trains the baseline CNN with AdamW on `dev_train` and validates on `dev_val`.

In [ ]:
!python run_baseline.py --dataset-dir dataset --features-csv extracted_features.csv --epochs 20

## 6. MOGEO Optimization (Feature Selection & CNN Hyperparameter Search)
Multi-Objective Golden Eagle Optimizer searches over feature selection flags (Shape, Size, Colour, Vein, Texture) and CNN architecture knobs to maximize validation accuracy and macro-F1.

In [ ]:
!python run_mogeo.py --dataset-dir dataset --features-csv extracted_features.csv --pop-size 8 --generations 6 --resume

## 7. Final Training with Best MOGEO Solution
Retrains the hybrid model using the best Pareto compromise solution found by MOGEO, creating `checkpoints/best.pt`.

In [ ]:
!python run_final.py --dataset-dir dataset --features-csv extracted_features.csv --epochs 40 --resume

## 8. Final Performance Evaluation (Untouched final_test Split)
Computes real Top-1 Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and Error Analysis.

In [ ]:
!python evaluate_final.py --dataset-dir dataset --features-csv extracted_features.csv

## 9. Predict Plant Species & Associated Medicinal Properties
Tests the final model on an unseen leaf image, outputs Top-5 predicted classes, and queries `properties.csv` for medicinal attributes.

In [ ]:
!python test_model.py --checkpoint checkpoints/best.pt --image "Amaranthus viridis (1).jpg" --properties-csv properties.csv